# Limpieza de la Base de Leads

Limpieza de `Analisis_GAC_Full7agosto_2026_Base_de_Leads_.csv`. A diferencia de las bases anteriores, aquí el problema principal no eran filas vacías sino texto con codificación dañada (acentos rotos) y una columna sin nombre.

## 1. Cargar los datos

In [2]:
import pandas as pd

# Ajusta esta ruta si el archivo no esta en la misma carpeta que este notebook
archivo_entrada = "Analisis GAC Full7agosto 2026(Base de Leads).csv"

df = pd.read_csv(archivo_entrada, encoding="utf-8-sig")

print("Forma original:", df.shape)
df.head()

Forma original: (210, 14)


,Fecha de Alta,Mes Lead,AgLead,Titulo,Nombre,Telefono,Correo,Producto,Temperatura,Comentario,Asesor,Fuente,Origen,
0,11/30/2025 8:38,11,834956f716e7ddae82527bf81c29ebc8,Lead ANGE,Apolonio Torres Perez,NaN,NaN,EMZOOM,Lead Muy Interesado,Se reactiva lead para captura de ventaFinal:EM...,CESAR DE JESUS PESTAÃ‘A,11.Origen Personalizado/Calle,PLAZA ANGELOPOLIS,1 Ventas
1,11/29/2025 21:17,11,13ee2f21ed8506825e8591bf1abdab51,Lead ANGE,Javier Nieto,NaN,NaN,GS4 MAX,Lead Interes Medio,SE LE CONTACTA EN PLAZA ANGELOPOLIS Final:GS4 ...,MAURICIO ALFREDO VAZQUEZ FARFAN,11.Origen Personalizado/Calle,PLAZA ANGELOPOLIS,Finalizado
2,11/29/2025 20:48,11,fad33b39f526b4ebe04a4b7e86002d5a,Lead ANGE,Ismael Segovia Banos,NaN,NaN,GS4 MAX,Lead Muy Interesado,SE INTERESA POR GS4 Final:GS4 MAX HEV,MAURICIO ALFREDO VAZQUEZ FARFAN,11.Origen Personalizado/Calle,PLAZA ANGELOPOLIS,1 Ventas
3,11/29/2025 16:11,11,e495cd540b7c7dc55b516fbdc7ea8bf0,Lead ANGE,Luis Jaime Estrada,NaN,NaN,EMKOO,Lead Muy Interesado,ofrecÃ­ tambiÃ©n GS4 y emko hv Pidieron:EMKOO HEV,RICARDO SILVA HERAS,11.Origen Personalizado/Calle,PLAZA ANGELOPOLIS,Contactado
4,11/29/2025 16:08,11,4d2b212032284fa42fa64a493e34c4ed,Lead ANGE,Hector Hamley,NaN,NaN,EMKOO,Lead Interes Medio,interÃ©s Pidieron:EMKOO HEV,RICARDO SILVA HERAS,11.Origen Personalizado/Calle,PLAZA ANGELOPOLIS,Contactado


## 2. Renombrar la columna sin nombre

La última columna del archivo no tiene encabezado (aparece como `" "`), pero sí trae datos reales: el estatus de cada lead (`Finalizado`, `Contactado`, `1 Ventas`). No es basura como las columnas "Items" de la Bitácora de Piso, así que en vez de eliminarla le ponemos un nombre.

In [3]:
df = df.rename(columns={" ": "Estatus"})
df["Estatus"].value_counts()

Estatus
Finalizado    123
Contactado     77
1 Ventas       10
Name: count, dtype: int64

## 3. Eliminar columnas vacías

`Telefono` y `Correo` no tienen ni un solo dato en las 210 filas.

In [4]:
print("Valores no nulos -> Telefono:", df["Telefono"].notna().sum(), "| Correo:", df["Correo"].notna().sum())

df = df.drop(columns=["Telefono", "Correo"])
print("Forma tras eliminar columnas:", df.shape)

Valores no nulos -> Telefono: 0 | Correo: 0
Forma tras eliminar columnas: (210, 12)


## 4. Corregir texto con codificación dañada

En `Asesor` y `Comentario` varios nombres y palabras con acento quedaron mal codificados durante alguna exportación anterior (ej. `"GonzÃ¡lez"` en vez de `"González"`, `"PESTAÃ‘A"` en vez de `"PESTAÑA"`). Es un patrón reconocible y reversible: se corrige solo en los valores que muestran ese patrón, sin tocar el resto del texto.

In [5]:
for columna in ["Asesor", "Comentario"]:
    con_mojibake = df[columna].str.contains("Ã", na=False)
    df.loc[con_mojibake, columna] = df.loc[con_mojibake, columna].str.encode("cp1252").str.decode("utf-8")

print("Asesores (ya corregidos):")
sorted(df["Asesor"].unique().tolist())

Asesores (ya corregidos):


['AURELIO FELIPE  TORRES REVILLA',
 'Alan González Belanzate',
 'Alvaro Celso Mora Rosete',
 'CESAR DE JESUS PESTAÑA',
 'IGNACIO ALEXANDRO ALDERETE GUEVARA',
 'MAURICIO ALFREDO VAZQUEZ FARFAN',
 'NANCY OLASCOAGA CASTILLO',
 'OMAR HERNANDEZ GARCIA',
 'PEDRO ANTONIO CINTO DE GANTE',
 'Pedro  Gonzáles Espíndol',
 'RICARDO SILVA HERAS']

## 5. Espacios sobrantes

Varios nombres y el título del lead traían espacios dobles o de más (ej. `"Lead  ANGE"`, `"Luis Jaime  Estrada   "`). Se recortan los espacios al inicio/final y se colapsan los espacios dobles a uno solo, en todas las columnas de texto.

In [6]:
for columna in df.columns:
    if df[columna].dtype == object or str(df[columna].dtype) == "str":
        df[columna] = df[columna].str.replace(r"\s+", " ", regex=True).str.strip()

print("Titulo:", df["Titulo"].unique().tolist())

Titulo: ['Lead ANGE', 'Lead ANGE - Return']


## 6. Rellenar los valores faltantes con "Sin dato"

In [7]:
nulos_antes = int(df.isna().sum().sum())
df = df.fillna("Sin dato")

print(f"Se rellenaron {nulos_antes} valores faltantes con 'Sin dato'")

Se rellenaron 14 valores faltantes con 'Sin dato'


## 7. Revisión final y exportar

In [8]:
print("Forma final:", df.shape)
print("Valores nulos restantes:", df.isna().sum().sum())
print("Filas duplicadas restantes:", df.duplicated().sum())

archivo_salida = "Base_de_Leads_Limpia.csv"
df.to_csv(archivo_salida, index=False, encoding="utf-8-sig")
print("Archivo guardado como:", archivo_salida)

df.head()

Forma final: (210, 12)
Valores nulos restantes: 0
Filas duplicadas restantes: 0
Archivo guardado como: Base_de_Leads_Limpia.csv


,Fecha de Alta,Mes Lead,AgLead,Titulo,Nombre,Producto,Temperatura,Comentario,Asesor,Fuente,Origen,Estatus
0,11/30/2025 8:38,11,834956f716e7ddae82527bf81c29ebc8,Lead ANGE,Apolonio Torres Perez,EMZOOM,Lead Muy Interesado,Se reactiva lead para captura de ventaFinal:EM...,CESAR DE JESUS PESTAÑA,11.Origen Personalizado/Calle,PLAZA ANGELOPOLIS,1 Ventas
1,11/29/2025 21:17,11,13ee2f21ed8506825e8591bf1abdab51,Lead ANGE,Javier Nieto,GS4 MAX,Lead Interes Medio,SE LE CONTACTA EN PLAZA ANGELOPOLIS Final:GS4 ...,MAURICIO ALFREDO VAZQUEZ FARFAN,11.Origen Personalizado/Calle,PLAZA ANGELOPOLIS,Finalizado
2,11/29/2025 20:48,11,fad33b39f526b4ebe04a4b7e86002d5a,Lead ANGE,Ismael Segovia Banos,GS4 MAX,Lead Muy Interesado,SE INTERESA POR GS4 Final:GS4 MAX HEV,MAURICIO ALFREDO VAZQUEZ FARFAN,11.Origen Personalizado/Calle,PLAZA ANGELOPOLIS,1 Ventas
3,11/29/2025 16:11,11,e495cd540b7c7dc55b516fbdc7ea8bf0,Lead ANGE,Luis Jaime Estrada,EMKOO,Lead Muy Interesado,ofrecí también GS4 y emko hv Pidieron:EMKOO HEV,RICARDO SILVA HERAS,11.Origen Personalizado/Calle,PLAZA ANGELOPOLIS,Contactado
4,11/29/2025 16:08,11,4d2b212032284fa42fa64a493e34c4ed,Lead ANGE,Hector Hamley,EMKOO,Lead Interes Medio,interés Pidieron:EMKOO HEV,RICARDO SILVA HERAS,11.Origen Personalizado/Calle,PLAZA ANGELOPOLIS,Contactado
